In [1]:
# Spotlight - Fairness Model
# Notebook 04

import pandas as pd
import numpy as np

print("Fairness model notebook ready!")

Fairness model notebook ready!


In [2]:
# Load the data required for the fairness model

small_matrix = pd.read_csv("../data/small_matrix.csv")
item_features = pd.read_csv("../data/item_daily_features.csv")

# Recreate the watch score used throughout the project
small_matrix["watch_score"] = (
    small_matrix["watch_ratio"].clip(upper=3.0)
)

# Remove interactions without a valid date
small_matrix = small_matrix.dropna(subset=["date"]).copy()

# Correctly parse KuaiRec's YYYYMMDD date format
small_matrix["date"] = pd.to_datetime(
    small_matrix["date"].astype(int).astype(str),
    format="%Y%m%d"
)

# Temporal split
train_end = pd.Timestamp("2020-08-29")

train_data = small_matrix[
    small_matrix["date"] <= train_end
].copy()

test_data = small_matrix[
    small_matrix["date"] > train_end
].copy()

print("Training interactions:", len(train_data))
print("Testing interactions:", len(test_data))
print("Training users:", train_data["user_id"].nunique())
print("Training videos:", train_data["video_id"].nunique())

Training interactions: 4387382
Testing interactions: 107196
Training users: 1411
Training videos: 3288


In [3]:
# Create the video-to-creator mapping

video_creator = (
    item_features[
        ["video_id", "author_id"]
    ]
    .dropna(subset=["video_id", "author_id"])
    .drop_duplicates("video_id")
    .copy()
)

print("Videos with creator mapping:", len(video_creator))
print("Unique creators:", video_creator["author_id"].nunique())

display(video_creator.head(10))

Videos with creator mapping: 10728
Unique creators: 8369


,video_id,author_id
0,0,3309
63,1,4978
126,2,939
189,3,5889
252,4,4284
304,5,3483
367,6,725
430,7,2742
492,8,3427
554,9,4523


In [4]:
# Restrict the creator population to videos used in our experiment

experiment_video_ids = small_matrix["video_id"].unique()

experiment_video_creator = (
    video_creator[
        video_creator["video_id"].isin(experiment_video_ids)
    ]
    .copy()
)

experiment_creators = (
    experiment_video_creator["author_id"]
    .unique()
)

print("Experiment videos:", len(experiment_video_ids))
print("Experiment creators:", len(experiment_creators))

print(
    "Creator mappings:",
    len(experiment_video_creator)
)

Experiment videos: 3327
Experiment creators: 2031
Creator mappings: 3327


In [5]:
# Calculate creator popularity from total historical show counts

creator_stats = (
    item_features
    .groupby("author_id")["show_cnt"]
    .sum()
    .reset_index(name="total_shows")
)

# Keep only creators represented in our experiment
creator_stats_exp = creator_stats[
    creator_stats["author_id"].isin(experiment_creators)
].copy()

print(
    "Creators in experiment:",
    len(creator_stats_exp)
)

print(
    "Average total shows:",
    creator_stats_exp["total_shows"].mean()
)

print(
    "Median total shows:",
    creator_stats_exp["total_shows"].median()
)

print(
    "Minimum total shows:",
    creator_stats_exp["total_shows"].min()
)

print(
    "Maximum total shows:",
    creator_stats_exp["total_shows"].max()
)

Creators in experiment: 2031
Average total shows: 14936152.600689316
Median total shows: 5574655.0
Minimum total shows: 150267
Maximum total shows: 375062603


In [6]:
# Define creator groups using the 25th percentile of total shows

creator_threshold = (
    creator_stats_exp["total_shows"]
    .quantile(0.25)
)

creator_stats_exp["creator_group"] = np.where(
    creator_stats_exp["total_shows"] <= creator_threshold,
    "Emerging",
    "Established"
)

print(
    "Emerging threshold:",
    creator_threshold
)

print("\nCreator counts:")
display(
    creator_stats_exp["creator_group"]
    .value_counts()
)

Emerging threshold: 1139461.5

Creator counts:


creator_group
Established    1523
Emerging        508
Name: count, dtype: int64

In [7]:
# Attach creator popularity and fairness group to each video

video_fairness = experiment_video_creator.merge(
    creator_stats_exp[
        ["author_id", "total_shows", "creator_group"]
    ],
    on="author_id",
    how="left"
)

print("Videos with fairness information:", len(video_fairness))
print("Unique videos:", video_fairness["video_id"].nunique())
print("Missing creator groups:", video_fairness["creator_group"].isna().sum())

display(
    video_fairness.head(10)
)

Videos with fairness information: 3327
Unique videos: 3327
Missing creator groups: 0


,video_id,author_id,total_shows,creator_group
0,103,1862,6886568,Established
1,109,1853,1338871,Established
2,120,4450,2649109,Established
3,122,888,39301005,Established
4,128,2269,27471863,Established
5,130,6512,24265619,Established
6,131,7420,26988517,Established
7,133,2624,1500282,Established
8,136,3542,2044055,Established
9,137,6045,10432120,Established


In [8]:
# Recreate the interest-aware user-video candidate scores

video_tags = (
    item_features[["video_id", "video_tag_name"]]
    .dropna(subset=["video_tag_name"])
    .drop_duplicates("video_id")
)

interaction_videos = small_matrix["video_id"].unique()

video_tags = video_tags[
    video_tags["video_id"].isin(interaction_videos)
].copy()

# Calculate smoothed user interest by original tag
user_history_tags = train_data.merge(
    video_tags,
    on="video_id",
    how="left"
).dropna(subset=["video_tag_name"])

user_interest = (
    user_history_tags
    .groupby(["user_id", "video_tag_name"])
    .agg(
        interactions=("video_id", "count"),
        avg_watch_score=("watch_score", "mean")
    )
    .reset_index()
)

# User-level average engagement
user_means = (
    train_data
    .groupby("user_id")["watch_score"]
    .mean()
    .rename("user_mean_watch_score")
    .reset_index()
)

user_interest = user_interest.merge(
    user_means,
    on="user_id",
    how="left"
)

# Same smoothing used in Notebook 03
smoothing = 5

user_interest["smoothed_interest"] = (
    (
        user_interest["interactions"]
        * user_interest["avg_watch_score"]
        + smoothing * user_interest["user_mean_watch_score"]
    )
    /
    (
        user_interest["interactions"] + smoothing
    )
)

print("Interest profile rows:", len(user_interest))
print("Users:", user_interest["user_id"].nunique())
print("Original tags:", user_interest["video_tag_name"].nunique())

Interest profile rows: 476973
Users: 1411
Original tags: 345


In [9]:
# Create all user-video candidate pairs for tagged videos

video_tag_lookup = (
    video_tags[
        ["video_id", "video_tag_name"]
    ]
    .drop_duplicates("video_id")
    .copy()
)

users = train_data["user_id"].unique()
videos = video_tag_lookup["video_id"].unique()

user_video_pairs = pd.MultiIndex.from_product(
    [users, videos],
    names=["user_id", "video_id"]
).to_frame(index=False)

print("Candidate user-video pairs:", len(user_video_pairs))
print("Users:", user_video_pairs["user_id"].nunique())
print("Videos:", user_video_pairs["video_id"].nunique())

Candidate user-video pairs: 4690164
Users: 1411
Videos: 3324


In [10]:
# Attach creator and fairness information to every candidate video

user_video_candidates = user_video_pairs.merge(
    video_fairness[
        ["video_id", "author_id", "total_shows", "creator_group"]
    ],
    on="video_id",
    how="left"
)

print("Candidate rows:", len(user_video_candidates))
print("Missing creator IDs:", user_video_candidates["author_id"].isna().sum())
print("Missing creator groups:", user_video_candidates["creator_group"].isna().sum())

print("\nCreator group distribution:")
display(
    user_video_candidates["creator_group"].value_counts()
)

Candidate rows: 4690164
Missing creator IDs: 0
Missing creator groups: 0

Creator group distribution:


creator_group
Established    3953622
Emerging        736542
Name: count, dtype: int64

In [11]:
# Attach each user's learned interest in the video's tag

user_video_candidates = user_video_candidates.merge(
    user_interest[
        [
            "user_id",
            "video_tag_name",
            "smoothed_interest"
        ]
    ],
    on=["user_id", "video_tag_name"],
    how="left"
)

# No learned interest evidence = 0
user_video_candidates["interest_relevance"] = (
    user_video_candidates["smoothed_interest"].fillna(0)
)

print("Candidate rows:", len(user_video_candidates))
print(
    "Non-zero interest scores:",
    (user_video_candidates["interest_relevance"] > 0).sum()
)
print(
    "Zero interest scores:",
    (user_video_candidates["interest_relevance"] == 0).sum()
)

KeyError: 'video_tag_name'

In [ ]:
# Attach the video's original tag to each candidate

user_video_candidates = user_video_candidates.merge(
    video_tag_lookup[
        ["video_id", "video_tag_name"]
    ],
    on="video_id",
    how="left"
)

print("Candidate rows:", len(user_video_candidates))
print(
    "Missing video tags:",
    user_video_candidates["video_tag_name"].isna().sum()
)

display(
    user_video_candidates.head(10)
)

Candidate rows: 4690164
Missing video tags: 0


,user_id,video_id,author_id,total_shows,creator_group,video_tag_name
0,14,103,1862,6886568,Established,面食
1,14,109,1853,1338871,Established,摩托车
2,14,120,4450,2649109,Established,搞笑
3,14,122,888,39301005,Established,猴
4,14,128,2269,27471863,Established,美食
5,14,130,6512,24265619,Established,侧面
6,14,131,7420,26988517,Established,电影黑条
7,14,133,2624,1500282,Established,猫
8,14,136,3542,2044055,Established,人群聚集
9,14,137,6045,10432120,Established,影视介绍


In [ ]:
# Attach each user's learned interest in the video's tag

user_video_candidates = user_video_candidates.merge(
    user_interest[
        [
            "user_id",
            "video_tag_name",
            "smoothed_interest"
        ]
    ],
    on=["user_id", "video_tag_name"],
    how="left"
)

# No learned interest evidence is treated as zero relevance
user_video_candidates["interest_relevance"] = (
    user_video_candidates["smoothed_interest"].fillna(0)
)

print("Candidate rows:", len(user_video_candidates))
print(
    "Non-zero interest scores:",
    (user_video_candidates["interest_relevance"] > 0).sum()
)
print(
    "Zero interest scores:",
    (user_video_candidates["interest_relevance"] == 0).sum()
)

Candidate rows: 4690164
Non-zero interest scores: 4678474
Zero interest scores: 11690


In [ ]:
# Normalize interest relevance separately for each user

user_video_candidates["interest_relevance_norm"] = (
    user_video_candidates
    .groupby("user_id")["interest_relevance"]
    .transform(
        lambda x: (
            (x - x.min()) / (x.max() - x.min())
            if x.max() > x.min()
            else 0
        )
    )
)

print(
    "Minimum normalized relevance:",
    user_video_candidates["interest_relevance_norm"].min()
)

print(
    "Maximum normalized relevance:",
    user_video_candidates["interest_relevance_norm"].max()
)

print("\nUser 14 top interest scores:")
display(
    user_video_candidates[
        user_video_candidates["user_id"] == 14
    ]
    .sort_values(
        "interest_relevance_norm",
        ascending=False
    )
    [["video_id", "video_tag_name",
      "creator_group",
      "interest_relevance",
      "interest_relevance_norm"]]
    .head(10)
)

Minimum normalized relevance: 0.0
Maximum normalized relevance: 1.0

User 14 top interest scores:


,video_id,video_tag_name,creator_group,interest_relevance,interest_relevance_norm
710,2117,孕妇知识,Established,1.378070,1.000000
2343,7157,孕妇知识,Established,1.378070,1.000000
1769,5374,孕妇知识,Established,1.378070,1.000000
2622,8200,穴位按摩,Established,1.306115,0.947786
3009,9594,穴位按摩,Established,1.306115,0.947786
3308,10410,穴位按摩,Established,1.306115,0.947786
1783,5421,聊天截屏,Established,1.290261,0.936281
534,1305,轮滑,Emerging,1.290261,0.936281
3197,10019,工作场所,Established,1.290261,0.936281
1489,4180,饮料,Established,1.285701,0.932972


In [ ]:
# Assign a controlled fairness boost to Emerging creators

fairness_boost = 0.10

user_video_candidates["fairness_boost"] = np.where(
    user_video_candidates["creator_group"] == "Emerging",
    fairness_boost,
    0
)

print("Fairness boost:", fairness_boost)

print("\nBoost distribution:")
display(
    user_video_candidates["fairness_boost"].value_counts()
)

Fairness boost: 0.1

Boost distribution:


fairness_boost
0.0    3953622
0.1     736542
Name: count, dtype: int64

In [ ]:
# Combine user relevance with the controlled fairness boost

user_video_candidates["fairness_score"] = (
    user_video_candidates["interest_relevance_norm"]
    + user_video_candidates["fairness_boost"]
)

print("Fairness score range:")
print(
    "Minimum:",
    user_video_candidates["fairness_score"].min()
)
print(
    "Maximum:",
    user_video_candidates["fairness_score"].max()
)

print("\nUser 14 top fairness-aware candidates:")

display(
    user_video_candidates[
        user_video_candidates["user_id"] == 14
    ]
    .sort_values(
        "fairness_score",
        ascending=False
    )
    [
        [
            "video_id",
            "video_tag_name",
            "creator_group",
            "interest_relevance_norm",
            "fairness_boost",
            "fairness_score"
        ]
    ]
    .head(10)
)


Fairness score range:
Minimum: 0.0
Maximum: 1.1

User 14 top fairness-aware candidates:


,video_id,video_tag_name,creator_group,interest_relevance_norm,fairness_boost,fairness_score
534,1305,轮滑,Emerging,0.936281,0.1,1.036281
710,2117,孕妇知识,Established,1.000000,0.0,1.000000
2343,7157,孕妇知识,Established,1.000000,0.0,1.000000
1769,5374,孕妇知识,Established,1.000000,0.0,1.000000
204,600,卖包,Emerging,0.882956,0.1,0.982956
929,2519,字幕遮脸,Emerging,0.868023,0.1,0.968023
2856,8736,篮球_篮球场,Emerging,0.866485,0.1,0.966485
381,832,篮球_篮球场,Emerging,0.866485,0.1,0.966485
2365,7190,篮球_篮球场,Emerging,0.866485,0.1,0.966485
2120,6222,篮球_篮球场,Emerging,0.866485,0.1,0.966485


In [ ]:
# Generate Top-10 fairness-aware recommendations for every user

fairness_recommendations = (
    user_video_candidates
    .sort_values(
        ["user_id", "fairness_score"],
        ascending=[True, False]
    )
    .groupby("user_id")
    .head(10)
    .reset_index(drop=True)
)

print("Users:", fairness_recommendations["user_id"].nunique())
print("Recommendation slots:", len(fairness_recommendations))
print("Recommendations per user:")
print(
    fairness_recommendations
    .groupby("user_id")
    .size()
    .value_counts()
    .sort_index()
)

Users: 1411
Recommendation slots: 14110
Recommendations per user:
10    1411
Name: count, dtype: int64


In [ ]:
# Measure recommendation exposure by creator group

fairness_exposure = (
    fairness_recommendations["creator_group"]
    .value_counts()
    .rename_axis("creator_group")
    .reset_index(name="recommendation_slots")
)

fairness_exposure["exposure_percentage"] = (
    fairness_exposure["recommendation_slots"]
    / len(fairness_recommendations)
    * 100
)

display(fairness_exposure)

,creator_group,recommendation_slots,exposure_percentage
0,Emerging,11218,79.503898
1,Established,2892,20.496102


In [ ]:
# Compare recommendation exposure with creator population representation

population_share = {
    "Emerging": 25.0,
    "Established": 75.0
}

fairness_exposure["population_percentage"] = (
    fairness_exposure["creator_group"]
    .map(population_share)
)

fairness_exposure["exposure_gap"] = (
    fairness_exposure["exposure_percentage"]
    - fairness_exposure["population_percentage"]
)

display(fairness_exposure)

,creator_group,recommendation_slots,exposure_percentage,population_percentage,exposure_gap
0,Emerging,11218,79.503898,25.0,54.503898
1,Established,2892,20.496102,75.0,-54.503898


In [ ]:
# Test different fairness boost strengths

fairness_strengths = [0.00, 0.02, 0.04, 0.06, 0.08, 0.10]

fairness_sensitivity_results = []

for strength in fairness_strengths:

    # Calculate score for this fairness strength
    test_score = (
        user_video_candidates["interest_relevance_norm"]
        + np.where(
            user_video_candidates["creator_group"] == "Emerging",
            strength,
            0
        )
    )

    # Get Top-10 per user
    temp_recommendations = (
        user_video_candidates.assign(
            test_score=test_score
        )
        .sort_values(
            ["user_id", "test_score"],
            ascending=[True, False]
        )
        .groupby("user_id")
        .head(10)
    )

    # Emerging exposure
    emerging_slots = (
        temp_recommendations["creator_group"]
        .eq("Emerging")
        .sum()
    )

    emerging_exposure = (
        emerging_slots
        / len(temp_recommendations)
        * 100
    )

    exposure_gap = (
        emerging_exposure - 25.0
    )

    fairness_sensitivity_results.append({
        "fairness_strength": strength,
        "emerging_slots": emerging_slots,
        "emerging_exposure": emerging_exposure,
        "exposure_gap": exposure_gap
    })

fairness_sensitivity_results = pd.DataFrame(
    fairness_sensitivity_results
)

display(fairness_sensitivity_results)

,fairness_strength,emerging_slots,emerging_exposure,exposure_gap
0,0.00,3947,27.973069,2.973069
1,0.02,6226,44.124734,19.124734
2,0.04,7624,54.032601,29.032601
3,0.06,8951,63.437279,38.437279
4,0.08,10152,71.948972,46.948972
5,0.10,11218,79.503898,54.503898


In [ ]:
# Evaluate recommendation relevance at different fairness strengths

fairness_relevance_results = []

for strength in fairness_strengths:

    # Calculate fairness-adjusted score
    test_score = (
        user_video_candidates["interest_relevance_norm"]
        + np.where(
            user_video_candidates["creator_group"] == "Emerging",
            strength,
            0
        )
    )

    # Generate Top-10 recommendations
    temp_recommendations = (
        user_video_candidates.assign(
            test_score=test_score
        )
        .sort_values(
            ["user_id", "test_score"],
            ascending=[True, False]
        )
        .groupby("user_id")
        .head(10)
    )

    # Evaluate against temporal test data
    test_by_user = (
        test_data
        .groupby("user_id")["video_id"]
        .apply(set)
        .to_dict()
    )

    user_hit_rates = []

    for user_id, recs in temp_recommendations.groupby("user_id"):

        recommended_videos = set(
            recs["video_id"]
        )

        actual_videos = test_by_user.get(
            user_id,
            set()
        )

        hits = len(
            recommended_videos.intersection(
                actual_videos
            )
        )

        hit_rate = hits / 10
        user_hit_rates.append(hit_rate)

    average_hit_rate = np.mean(user_hit_rates)

    fairness_relevance_results.append({
        "fairness_strength": strength,
        "average_hit_rate_at_10": average_hit_rate
    })

fairness_relevance_results = pd.DataFrame(
    fairness_relevance_results
)

display(fairness_relevance_results)

,fairness_strength,average_hit_rate_at_10
0,0.00,0.014174
1,0.02,0.011198
2,0.04,0.009284
3,0.06,0.007583
4,0.08,0.006237
5,0.10,0.005315


In [ ]:
# Identify videos each user has already watched during training

user_watched_videos = (
    train_data
    .groupby("user_id")["video_id"]
    .apply(set)
    .to_dict()
)

print(
    "Users with training history:",
    len(user_watched_videos)
)

print(
    "User 14 training-watched videos:",
    len(user_watched_videos[14])
)

Users with training history: 1411
User 14 training-watched videos: 3158


In [ ]:
# Remove videos already watched by each user during training

user_video_candidates["already_watched"] = [
    video_id in user_watched_videos[user_id]
    for user_id, video_id
    in zip(
        user_video_candidates["user_id"],
        user_video_candidates["video_id"]
    )
]

recommendation_candidates = (
    user_video_candidates[
        ~user_video_candidates["already_watched"]
    ]
    .copy()
)

print(
    "Original candidate pairs:",
    len(user_video_candidates)
)

print(
    "Remaining recommendation candidates:",
    len(recommendation_candidates)
)

print(
    "Removed previously watched pairs:",
    len(user_video_candidates)
    - len(recommendation_candidates)
)

print(
    "Users remaining:",
    recommendation_candidates["user_id"].nunique()
)

Original candidate pairs: 4690164
Remaining recommendation candidates: 306863
Removed previously watched pairs: 4383301
Users remaining: 1411


In [ ]:
# Re-evaluate fairness strengths using only unseen videos

fairness_corrected_results = []

# Test-set videos for each user
test_by_user = (
    test_data
    .groupby("user_id")["video_id"]
    .apply(set)
    .to_dict()
)

for strength in fairness_strengths:

    # Interest relevance + Emerging creator boost
    test_score = (
        recommendation_candidates["interest_relevance_norm"]
        + np.where(
            recommendation_candidates["creator_group"] == "Emerging",
            strength,
            0
        )
    )

    # Generate Top-10 recommendations
    temp_recommendations = (
        recommendation_candidates
        .assign(test_score=test_score)
        .sort_values(
            ["user_id", "test_score"],
            ascending=[True, False]
        )
        .groupby("user_id")
        .head(10)
    )

    # Calculate HitRate@10
    user_hit_rates = []

    for user_id, recs in temp_recommendations.groupby("user_id"):

        recommended_videos = set(
            recs["video_id"]
        )

        actual_videos = test_by_user.get(
            user_id,
            set()
        )

        hits = len(
            recommended_videos.intersection(
                actual_videos
            )
        )

        user_hit_rates.append(hits / 10)

    average_hit_rate = np.mean(user_hit_rates)

    # Calculate Emerging exposure
    emerging_slots = (
        temp_recommendations["creator_group"]
        .eq("Emerging")
        .sum()
    )

    emerging_exposure = (
        emerging_slots
        / len(temp_recommendations)
        * 100
    )

    exposure_gap = (
        emerging_exposure - 25.0
    )

    fairness_corrected_results.append({
        "fairness_strength": strength,
        "emerging_exposure": emerging_exposure,
        "exposure_gap": exposure_gap,
        "average_hit_rate_at_10": average_hit_rate
    })

fairness_corrected_results = pd.DataFrame(
    fairness_corrected_results
)

display(fairness_corrected_results)

,fairness_strength,emerging_exposure,exposure_gap,average_hit_rate_at_10
0,0.00,7.335223,-17.664777,0.438767
1,0.02,11.956060,-13.043940,0.420269
2,0.04,15.889440,-9.110560,0.406662
3,0.06,19.971651,-5.028349,0.390645
4,0.08,25.279943,0.279943,0.371297
5,0.10,31.098512,6.098512,0.352020


In [ ]:
# Measure creator coverage under each fairness strength

fairness_coverage_results = []

total_emerging_creators = (
    creator_stats_exp["creator_group"]
    .eq("Emerging")
    .sum()
)

total_established_creators = (
    creator_stats_exp["creator_group"]
    .eq("Established")
    .sum()
)

for strength in fairness_strengths:

    test_score = (
        recommendation_candidates["interest_relevance_norm"]
        + np.where(
            recommendation_candidates["creator_group"] == "Emerging",
            strength,
            0
        )
    )

    temp_recommendations = (
        recommendation_candidates
        .assign(test_score=test_score)
        .sort_values(
            ["user_id", "test_score"],
            ascending=[True, False]
        )
        .groupby("user_id")
        .head(10)
    )

    exposed_creators = (
        temp_recommendations[
            ["author_id", "creator_group"]
        ]
        .drop_duplicates()
    )

    emerging_exposed = (
        exposed_creators["creator_group"]
        .eq("Emerging")
        .sum()
    )

    established_exposed = (
        exposed_creators["creator_group"]
        .eq("Established")
        .sum()
    )

    fairness_coverage_results.append({
        "fairness_strength": strength,
        "emerging_creators_exposed": emerging_exposed,
        "emerging_creator_coverage": (
            emerging_exposed
            / total_emerging_creators
            * 100
        ),
        "established_creators_exposed": established_exposed,
        "established_creator_coverage": (
            established_exposed
            / total_established_creators
            * 100
        )
    })

fairness_coverage_results = pd.DataFrame(
    fairness_coverage_results
)

display(fairness_coverage_results)

,fairness_strength,emerging_creators_exposed,emerging_creator_coverage,established_creators_exposed,established_creator_coverage
0,0.00,198,38.976378,755,49.573211
1,0.02,237,46.653543,742,48.719632
2,0.04,265,52.165354,735,48.260013
3,0.06,284,55.905512,713,46.815496
4,0.08,313,61.614173,694,45.567958
5,0.10,346,68.110236,681,44.714380


In [ ]:
# Measure creator diversity in the Top-10 recommendations

creator_diversity_results = []

for strength in fairness_strengths:

    test_score = (
        recommendation_candidates["interest_relevance_norm"]
        + np.where(
            recommendation_candidates["creator_group"] == "Emerging",
            strength,
            0
        )
    )

    temp_recommendations = (
        recommendation_candidates
        .assign(test_score=test_score)
        .sort_values(
            ["user_id", "test_score"],
            ascending=[True, False]
        )
        .groupby("user_id")
        .head(10)
    )

    unique_creators_per_user = (
        temp_recommendations
        .groupby("user_id")["author_id"]
        .nunique()
    )

    creator_diversity_results.append({
        "fairness_strength": strength,
        "mean_unique_creators_at_10": (
            unique_creators_per_user.mean()
        ),
        "median_unique_creators_at_10": (
            unique_creators_per_user.median()
        ),
        "minimum_unique_creators_at_10": (
            unique_creators_per_user.min()
        ),
        "maximum_unique_creators_at_10": (
            unique_creators_per_user.max()
        )
    })

creator_diversity_results = pd.DataFrame(
    creator_diversity_results
)

display(creator_diversity_results)

,fairness_strength,mean_unique_creators_at_10,median_unique_creators_at_10,minimum_unique_creators_at_10,maximum_unique_creators_at_10
0,0.00,9.675408,10.0,5,10
1,0.02,9.697378,10.0,5,10
2,0.04,9.715804,10.0,5,10
3,0.06,9.733522,10.0,5,10
4,0.08,9.751240,10.0,5,10
5,0.10,9.776754,10.0,6,10


In [ ]:
# Combine fairness, relevance, coverage, and diversity results

fairness_summary = (
    fairness_corrected_results
    .merge(
        fairness_coverage_results,
        on="fairness_strength"
    )
    .merge(
        creator_diversity_results,
        on="fairness_strength"
    )
)

display(fairness_summary)

,fairness_strength,emerging_exposure,exposure_gap,average_hit_rate_at_10,emerging_creators_exposed,emerging_creator_coverage,established_creators_exposed,established_creator_coverage,mean_unique_creators_at_10,median_unique_creators_at_10,minimum_unique_creators_at_10,maximum_unique_creators_at_10
0,0.00,7.335223,-17.664777,0.438767,198,38.976378,755,49.573211,9.675408,10.0,5,10
1,0.02,11.956060,-13.043940,0.420269,237,46.653543,742,48.719632,9.697378,10.0,5,10
2,0.04,15.889440,-9.110560,0.406662,265,52.165354,735,48.260013,9.715804,10.0,5,10
3,0.06,19.971651,-5.028349,0.390645,284,55.905512,713,46.815496,9.733522,10.0,5,10
4,0.08,25.279943,0.279943,0.371297,313,61.614173,694,45.567958,9.751240,10.0,5,10
5,0.10,31.098512,6.098512,0.352020,346,68.110236,681,44.714380,9.776754,10.0,6,10


In [ ]:
# Combine fairness, relevance, coverage, and creator-diversity results

fairness_summary = (
    fairness_corrected_results
    .merge(
        fairness_coverage_results,
        on="fairness_strength"
    )
    .merge(
        creator_diversity_results,
        on="fairness_strength"
    )
)

display(fairness_summary)

,fairness_strength,emerging_exposure,exposure_gap,average_hit_rate_at_10,emerging_creators_exposed,emerging_creator_coverage,established_creators_exposed,established_creator_coverage,mean_unique_creators_at_10,median_unique_creators_at_10,minimum_unique_creators_at_10,maximum_unique_creators_at_10
0,0.00,7.335223,-17.664777,0.438767,198,38.976378,755,49.573211,9.675408,10.0,5,10
1,0.02,11.956060,-13.043940,0.420269,237,46.653543,742,48.719632,9.697378,10.0,5,10
2,0.04,15.889440,-9.110560,0.406662,265,52.165354,735,48.260013,9.715804,10.0,5,10
3,0.06,19.971651,-5.028349,0.390645,284,55.905512,713,46.815496,9.733522,10.0,5,10
4,0.08,25.279943,0.279943,0.371297,313,61.614173,694,45.567958,9.751240,10.0,5,10
5,0.10,31.098512,6.098512,0.352020,346,68.110236,681,44.714380,9.776754,10.0,6,10


In [12]:
# Attach creator and fairness information to every candidate video

user_video_candidates = user_video_pairs.merge(
    video_fairness[
        ["video_id", "author_id", "total_shows", "creator_group"]
    ],
    on="video_id",
    how="left"
)

print("Candidate rows:", len(user_video_candidates))
print("Missing creator IDs:", user_video_candidates["author_id"].isna().sum())
print("Missing creator groups:", user_video_candidates["creator_group"].isna().sum())

print("\nCreator group distribution:")
display(
    user_video_candidates["creator_group"].value_counts()
)

Candidate rows: 4690164
Missing creator IDs: 0
Missing creator groups: 0

Creator group distribution:


creator_group
Established    3953622
Emerging        736542
Name: count, dtype: int64

## Notebook 04 — Fairness Model Findings

The fairness-aware recommender was evaluated using six fairness-strength values
from 0.00 to 0.10. The fairness strength adds an exposure boost to videos
belonging to Emerging creators while retaining the learned user-interest
relevance score.

As fairness strength increased, Emerging-creator exposure increased from
7.34% at strength 0.00 to 31.10% at strength 0.10. Emerging-creator coverage
also increased from 38.98% to 68.11%.

At fairness strength 0.08, Emerging creators received 25.28% of recommendation
slots, compared with their 25% representation in the creator population.
The resulting exposure gap was +0.28 percentage points.

However, increasing the fairness strength was associated with a reduction in
HitRate@10, from 43.88% at strength 0.00 to 35.20% at strength 0.10. This
demonstrates a relevance–fairness trade-off in the current re-ranking
mechanism.

Creator diversity changed only slightly, with mean unique creators per Top-10
increasing from 9.68 at strength 0.00 to 9.78 at strength 0.10.

These results establish that creator exposure can be systematically modified
through fairness-aware re-ranking, while also showing that stronger exposure
intervention can affect recommendation relevance. The subsequent exploration
strategy will investigate whether controlled exploration can introduce
additional creator/content diversity while retaining user-interest signals.